# 16 · Sistemas de recomendación

Los recomendadores combinan Machine Learning, ranking y producto. El objetivo no siempre es predecir ratings: normalmente queremos **ordenar** ítems relevantes para cada usuario bajo restricciones de novedad, diversidad y negocio.

## Objetivos
- Construir baselines de popularidad.
- Implementar content-based con similitud coseno.
- Entender collaborative filtering user-item.
- Implementar matrix factorization sencilla.
- Evaluar ranking con Precision@K, Recall@K y NDCG.
- Entender cold start, implicit feedback, diversidad y sesgos de exposición.


In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
SEED=42; rng=np.random.default_rng(SEED)
# catálogo sintético
items=pd.DataFrame({'item':range(30),'desc':[f'{a} {b}' for a,b in zip(rng.choice(['ia','datos','python','cloud','ml'],30),rng.choice(['basico','practico','avanzado','proyecto'],30))]})
users=range(80); rows=[]
for u in users:
 pref=rng.choice(range(30),size=rng.integers(4,12),replace=False)
 for i in pref: rows.append([u,i,int(rng.integers(1,6))])
ratings=pd.DataFrame(rows,columns=['user','item','rating']); ratings.head()

## 1. Popularidad: baseline obligatorio
Una lista global por cantidad de interacciones o rating promedio es difícil de superar para usuarios nuevos y sirve para medir cuánto aporta la personalización. Hay que regularizar ratings de ítems con pocos votos.


In [ ]:
pop=ratings.groupby('item').agg(n=('rating','size'),mean=('rating','mean')); pop['score']=pop['mean']*pop['n']/(pop['n']+5)+ratings.rating.mean()*5/(pop['n']+5); display(pop.sort_values('score',ascending=False).head(10))

## 2. Content-based
Representamos ítems por atributos o embeddings y recomendamos ítems similares a lo que el usuario consumió. Ventaja: funciona con ítems nuevos si conocemos contenido. Limitación: puede crear una burbuja demasiado parecida a lo ya visto.


In [ ]:
tf=TfidfVectorizer(); E=tf.fit_transform(items.desc)
def content_recommend(item_id,k=5):
 sim=cosine_similarity(E[item_id],E).ravel(); idx=np.argsort(-sim); idx=idx[idx!=item_id][:k]; return items.iloc[idx].assign(sim=sim[idx])
display(items.iloc[[0]]); display(content_recommend(0))

## 3. Collaborative filtering
La señal está en patrones de comportamiento: usuarios que consumieron cosas similares tienden a compartir preferencias. La matriz usuario×ítem suele ser muy sparse.

Matrix factorization aproxima $R\approx UV^T$, donde cada usuario e ítem obtiene un vector latente. En implicit feedback se modelan clics/compras/tiempo, a menudo con weighted matrix factorization o BPR.


In [ ]:
R=ratings.pivot_table(index='user',columns='item',values='rating').fillna(0)
svd=TruncatedSVD(n_components=8,random_state=SEED); U=svd.fit_transform(R); V=svd.components_.T
scores=U@V.T
def recommend_user(u,k=5):
 seen=set(ratings.loc[ratings.user==u,'item']); order=np.argsort(-scores[u]); rec=[i for i in order if i not in seen][:k]; return items.set_index('item').loc[rec].assign(score=scores[u,rec])
display(recommend_user(0))

## 4. Evaluación offline temporal
No uses interacciones futuras para entrenar recomendaciones pasadas. En datasets reales, split temporal por usuario es más realista. Métricas:
- Precision@K: proporción recomendada que resultó relevante.
- Recall@K: proporción de relevantes recuperados.
- MAP@K y NDCG@K: consideran ranking.
- Coverage, diversity, novelty, serendipity: salud del catálogo.

Offline score no garantiza impacto: la evaluación final suele requerir A/B test.


In [ ]:
def precision_recall_at_k(recommended,relevant,k):
 rec=recommended[:k]; hits=len(set(rec)&set(relevant)); return hits/k, hits/max(1,len(relevant))
print(precision_recall_at_k([1,2,3,4,5],[2,5,8],5))

## 5. Arquitecturas modernas
- implicit ALS;
- BPR pairwise ranking;
- factorization machines;
- two-tower retrieval;
- deep ranking models;
- sequence recommenders (SASRec/transformers);
- embeddings multimodales;
- retrieval + reranking;
- bandits para exploration/exploitation.

## 6. Problemas reales
**Cold start:** usuario o ítem nuevo. Solución: popularidad contextual, contenido, onboarding.
**Feedback loops:** solo observamos feedback de lo que mostramos.
**Popularity bias:** los populares reciben más exposición y se vuelven aún más populares.
**Diversity:** top scores pueden ser casi idénticos.
**Fairness:** proveedores/grupos pueden recibir exposición desigual.

## Ejercicios
1. Implementa user-user y item-item cosine CF.
2. Haz matrix factorization con SGD desde cero.
3. Construye split leave-one-out por usuario.
4. Calcula NDCG@10.
5. Añade penalización de popularidad para aumentar novedad.
6. Implementa Maximum Marginal Relevance para diversidad.
7. Investiga `implicit`, LightFM y two-tower retrieval con TensorFlow/PyTorch.
